# Monthyl Core HR metrics

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

## Data preparation

In [2]:
fields = [
    'ds_start', 'ds', 'employee_id', 'hire_date', 'prehire_status', 'is_active',
    'termination_date', 'termination_reason', 'is_termination_voluntary', 'is_terminated',
    'org_l00', 'org_l01', 'org_l02', 'org_l03', 
    'gender', 'gender_remapped', 'ethnicity', 'ethnicity_remapped', 
    'is_manager_track', 'job_track', 'job_level_idx',
    'job_level_category', 'job_level_category_ordered_w_indicators'
]

In [3]:
df_employees_plus = pd.read_csv(
    filepath_or_buffer='../transforms/exclude/employee_data_plus.csv',
    dtype='str'
)
# type casting and renaming fields
df_employees_plus['ds'] = pd.to_datetime(df_employees_plus['full_date']).dt.normalize()
df_employees_plus['ds_start'] = df_employees_plus.ds.dt.to_period('M').dt.start_time
df_employees_plus['hire_date'] = pd.to_datetime(df_employees_plus['hire_date']).dt.normalize()
df_employees_plus['termination_date'] = pd.to_datetime(df_employees_plus['termination_date_coalesced']).dt.normalize()
df_employees_plus['is_termination_voluntary'] = df_employees_plus.is_termination_voluntary.astype('bool')
df_employees_plus['is_manager_track'] = df_employees_plus.is_manager_track.astype('bool')

# deriving is_active
condition_active = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds <= df_employees_plus.termination_date)
df_employees_plus['is_active'] = condition_active

# deriving is_terminated
condition_terminated_in_current_month = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds_start <= df_employees_plus.termination_date) &\
    (df_employees_plus.ds >= df_employees_plus.termination_date)
df_employees_plus['is_terminated'] = condition_terminated_in_current_month

# reorganizing fields
df_employees_plus = df_employees_plus[fields]

df_employees_plus.dtypes

ds_start                                   datetime64[ns]
ds                                         datetime64[ns]
employee_id                                        object
hire_date                                  datetime64[ns]
prehire_status                                     object
is_active                                            bool
termination_date                           datetime64[ns]
termination_reason                                 object
is_termination_voluntary                             bool
is_terminated                                        bool
org_l00                                            object
org_l01                                            object
org_l02                                            object
org_l03                                            object
gender                                             object
gender_remapped                                    object
ethnicity                                          object
ethnicity_rema

In [4]:
df_employees_plus.head()

,ds_start,ds,employee_id,hire_date,prehire_status,is_active,termination_date,termination_reason,is_termination_voluntary,is_terminated,...,org_l03,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators
0,2021-01-01,2021-01-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
1,2021-02-01,2021-02-28,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
2,2021-03-01,2021-03-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
3,2021-04-01,2021-04-30,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
4,2021-05-01,2021-05-31,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,False,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)


## Monthly results

### Creating monthly dataframe

In [5]:
df_monthly = pd.DataFrame({'ds': df_employees_plus.ds.unique()})

### Deriving `n_active_employees`

In [6]:
# df_monthly['n_active_employees_by_status']
active_employees_by_status = df_employees_plus[df_employees_plus['is_active']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_monthly = df_monthly.merge(right=active_employees_by_status, how='left', on='ds')

### Deriving `n_monthly_terminated_employees`

In [7]:
terminated_by_dates_monthly_by_status = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees')

df_monthly = df_monthly.merge(right=terminated_by_dates_monthly_by_status, how='left', on='ds')
df_monthly['n_monthly_terminated_employees'] = pd.to_numeric(df_monthly.n_monthly_terminated_employees, errors='coerce').fillna(0).astype('int')


### Deriving fields for `attrition_rate`

- `avg_active_employees` - active employee count 2 month rolling (between current month and previous month)
- `attrition_rate = terminated_employees / avg_active_employees`

In [8]:
df_monthly['avg_active_employees'] = df_monthly['n_active_employees'].rolling(window=2).mean()
df_monthly['avg_active_employees'] = np.where(
    df_monthly['avg_active_employees'].isnull(),
    df_monthly['n_active_employees'],
    df_monthly['avg_active_employees']
)
df_monthly['attrition_rate'] = df_monthly['n_monthly_terminated_employees'] / df_monthly['avg_active_employees']

## Monthly results by organization

### Deriving counts for active and terminated employees

In [9]:
indices = ['ds', 'org_l00', 'org_l01', 'org_l02']

# deriving active employee count
df_monthly_by_org = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(indices)['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees') \

df_monthly_by_org = pd.merge(left=df_monthly_by_org, right=terminated_by_dates_monthly_by_status_and_org, how='left', on=indices)
# df_monthly_by_org['n_monthly_terminated_employees'] = df_monthly.n_monthly_terminated_employees.fillna(0).astype('int')
df_monthly_by_org['n_monthly_terminated_employees'] = pd.to_numeric(df_monthly_by_org['n_monthly_terminated_employees'], errors='coerce')
df_monthly_by_org['n_monthly_terminated_employees'] = df_monthly_by_org['n_monthly_terminated_employees'].fillna(0).astype('int')


In [10]:
print(terminated_by_dates_monthly_by_status_and_org.dtypes)
terminated_by_dates_monthly_by_status_and_org[
    (terminated_by_dates_monthly_by_status_and_org.org_l02 == 'People')
    & (terminated_by_dates_monthly_by_status_and_org.ds >= '2021-01-01')
    & (terminated_by_dates_monthly_by_status_and_org.ds <= '2021-12-31')
]

ds                                datetime64[ns]
org_l00                                   object
org_l01                                   object
org_l02                                   object
n_monthly_terminated_employees             int64
dtype: object


,ds,org_l00,org_l01,org_l02,n_monthly_terminated_employees
3,2021-01-31,Company Inc.,Administrative,People,3
19,2021-03-31,Company Inc.,Administrative,People,4
42,2021-06-30,Company Inc.,Administrative,People,1
53,2021-07-31,Company Inc.,Administrative,People,1
78,2021-09-30,Company Inc.,Administrative,People,2
92,2021-10-31,Company Inc.,Administrative,People,1
105,2021-11-30,Company Inc.,Administrative,People,1
116,2021-12-31,Company Inc.,Administrative,People,1


In [11]:
print(df_monthly_by_org.dtypes)
df_monthly_by_org[
    (df_monthly_by_org.org_l02 == 'People')
    & (df_monthly_by_org.ds >= '2021-01-01')
    & (df_monthly_by_org.ds <= '2021-12-31')
]

ds                                datetime64[ns]
org_l00                                   object
org_l01                                   object
org_l02                                   object
n_active_employees                         int64
n_monthly_terminated_employees             int64
dtype: object


,ds,org_l00,org_l01,org_l02,n_active_employees,n_monthly_terminated_employees
4,2021-01-31,Company Inc.,Administrative,People,66,3
20,2021-02-28,Company Inc.,Administrative,People,67,0
36,2021-03-31,Company Inc.,Administrative,People,64,4
52,2021-04-30,Company Inc.,Administrative,People,71,0
68,2021-05-31,Company Inc.,Administrative,People,73,0
84,2021-06-30,Company Inc.,Administrative,People,75,1
100,2021-07-31,Company Inc.,Administrative,People,74,1
116,2021-08-31,Company Inc.,Administrative,People,74,0
132,2021-09-30,Company Inc.,Administrative,People,74,2
148,2021-10-31,Company Inc.,Administrative,People,76,1


### Deriving two-month average of active employees

In [12]:
df_reindexed = df_monthly_by_org \
    .set_index(indices).sort_index(level='ds') \
    .copy()
df_avg_active_employees = df_reindexed.groupby(level=indices[1:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0,1,2], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_org = pd.merge(left=df_monthly_by_org, right=df_avg_active_employees, how='left', on=indices)
df_monthly_by_org['avg_active_employees'] = np.where(
    df_monthly_by_org['avg_active_employees'].isnull(),
    df_monthly_by_org['n_active_employees'],
    df_monthly_by_org['avg_active_employees']
)

### Deriving `attrition_rate`

In [13]:
df_monthly_by_org['attrition_rate'] = df_monthly_by_org['n_monthly_terminated_employees'] / df_monthly_by_org['avg_active_employees']

## Analysis

<summary>Monthly attrition significance POC (proof of concept)</summary>
<details>
    <code>
        df_2021_01_31 = df_monthly_by_org[
            (df_monthly_by_org.ds == '2021-01-31')
        ].sort_values('org_l02') \
            [['ds', 'org_l01', 'org_l02', 'n_active_employees', 'n_monthly_terminated_employees', 'attrition_rate']] \
            .reset_index(drop=True)
        df_2021_01_31['n_not_terminated_employees'] = df_2021_01_31['n_active_employees'] - df_2021_01_31['n_monthly_terminated_employees']

        total_terminated = df_2021_01_31['n_monthly_terminated_employees'].sum()
        total_not_terminated = df_2021_01_31['n_not_terminated_employees'].sum()

        results = []

        for index, row in df_2021_01_31.iterrows():
            # for org under review
            org_name = row['org_l02']
            terminated_org = row['n_monthly_terminated_employees']
            not_terminated_org = row['n_not_terminated_employees']
            
            # for other orgs
            terminated_rest = total_terminated - terminated_org
            not_terminated_rest = total_not_terminated - not_terminated_org
            
            # 2x2 contingency table
            # (org vs others) x (terminated vs not terminated)
            table = [[terminated_org, not_terminated_org],
                    [terminated_rest, not_terminated_rest]]
            
            odds_ratio, p_value = fisher_exact(table)
            
            results.append({
                'org_l02': org_name,
                'contingency_table': str(table),
                'odds_ratio': odds_ratio,
                'p_value': p_value
            })
            
        results_df = pd.DataFrame(results)
        results_df.sort_values(by='p_value')
    </code>
</details>

In [36]:
df_monthly_by_org_l02 = df_monthly_by_org.copy(True)

df_monthly_by_org_l02['n_not_terminated_employees'] = df_monthly_by_org_l02['n_active_employees'] - df_monthly_by_org_l02['n_monthly_terminated_employees']

df_monthly_by_org_l02['contingency_table'] = ''
df_monthly_by_org_l02['odds_ratio'] = np.NaN
df_monthly_by_org_l02['p_value'] = np.NaN

for ds in df_monthly_by_org_l02.ds.unique():
    df_current = df_monthly_by_org_l02[df_monthly_by_org_l02.ds == ds].copy(True)
    total_terminated = df_current['n_monthly_terminated_employees'].sum()
    total_not_terminated = df_current['n_not_terminated_employees'].sum()

    for index, row in df_current.iterrows():
        # for org under review
        ds = row['ds']
        org_name = row['org_l02']
        terminated_org = row['n_monthly_terminated_employees']
        not_terminated_org = row['n_not_terminated_employees']
        
        # for other orgs
        terminated_rest = total_terminated - terminated_org
        not_terminated_rest = total_not_terminated - not_terminated_org
        
        # 2x2 contingency table
        # (org vs others) x (terminated vs not terminated)
        table = [[terminated_org, not_terminated_org],
                [terminated_rest, not_terminated_rest]]
        
        odds_ratio, p_value = fisher_exact(table)
        
        contingency_table = str(table)
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'contingency_table'
        ] = str(table)
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'odds_ratio'
        ] = odds_ratio
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'p_value'
        ] = p_value
df_monthly_by_org_l02['is_significant'] = df_monthly_by_org_l02['p_value'] <= 0.05

In [44]:
ds = '2022-02-28'
df_monthly_by_org_l02 \
    [df_monthly_by_org_l02.ds == ds] \
    [['ds', 'org_l00', 'org_l01', 'org_l02', 'attrition_rate', 'p_value', 'is_significant']] \
    .sort_values(['org_l00', 'org_l01', 'org_l02']) \
    .reset_index(drop=True) \
    .style.format({
        'ds': lambda t: t.strftime('%Y-%m-%d') if pd.notnull(t) else '',
        'attrition_rate': '{:.1%}',
        'p_value': '{:.2f}'
    })

,ds,org_l00,org_l01,org_l02,attrition_rate,p_value,is_significant
0,2022-02-28,Company Inc.,Administrative,Communications,8.9%,0.23,False
1,2022-02-28,Company Inc.,Administrative,Finance,4.6%,0.80,False
2,2022-02-28,Company Inc.,Administrative,IT Services,12.3%,0.03,True
3,2022-02-28,Company Inc.,Administrative,Legal,8.2%,0.59,False
4,2022-02-28,Company Inc.,Administrative,People,2.6%,0.23,False
5,2022-02-28,Company Inc.,Administrative,Risk Management,5.3%,1.00,False
6,2022-02-28,Company Inc.,Production,Hardware,2.1%,0.08,False
7,2022-02-28,Company Inc.,Production,Quality Control,8.0%,0.49,False
8,2022-02-28,Company Inc.,Production,Research,2.5%,0.23,False
9,2022-02-28,Company Inc.,Production,Service Delivery,6.1%,1.00,False
